In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
train_df = pd.read_csv('data/titanic_train4.csv')
test_df = pd.read_csv('data/titanic_test4.csv')

In [3]:
train_df.columns

Index(['Survived', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'isChild',
       'FamilySize', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C',
       'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr',
       'Title_Mrs', 'Title_Rare', 'FamilySizeGroup_0', 'FamilySizeGroup_1',
       'FamilySizeGroup_2'],
      dtype='str')

### Sex

In [4]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱 검증
# 교차표 생성
crosstab = pd.crosstab(train_df['Sex'], train_df['Survived'])

# 카이제곱 검증 수행
# 카이제곱 통계랑, p-value, 자유도, 기대빈도
_, p_value, _, _ = chi2_contingency(crosstab)
print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Sex와 Survived는 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다
- Sex와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Age

In [5]:
# 연속형 수치형 데이터 vs 범주형 데이터 → 포인트 이분 상관계수
_, p_value, = pointbiserialr(train_df['Age'], train_df['Survived'])
print(f'p-value : {p_value:.4f}')

p-value : 0.0528


- p-value가 0.05보다 크므로 Age와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각하지 못한다.
- Age와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 없다.

### SibSp

In [6]:
# 연속형 수치형 데이터 vs 범주형 데이터  → 포인트 이분 상관계수
_, p_value = pointbiserialr(train_df['SibSp'], train_df['Survived'])
print(f'p-value : {p_value:.4f}')

p-value : 0.2922


- p-value가 0.05보다 훨씬 크므로 SibSp와 Sruvived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각하지 못한다.
- SibSp와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 없다.

### Parch

In [7]:
# 연속형 수치형 데이터 vs 범주형 데이터 → 포인트 이분 상관계수
_, p_value = pointbiserialr(train_df['Parch'], train_df['Survived'])
print(f'p-value : {p_value:.4f}')

p-value : 0.0148


- p-value가 0.05보다 작으므로 Parch는 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Parch와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다

### Fare

In [8]:
# 연속형 수치형 데이터 vs 범주형 데이터 → 포인트 이분 상관계수
_, p_value = pointbiserialr(train_df['Fare'], train_df['Survived'])
print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Fare와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다
- Fare와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### isChild

In [9]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱
crosstab = pd.crosstab(train_df['isChild'], train_df['Survived'])
_,p_value, _, _ =chi2_contingency(crosstab)
print(f'p-value : {p_value:.4f}')

p-value : 0.0003


- p-value가 0.05보다 작으므로 isChild Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- isChild와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다

### FamilySize

In [10]:
# 연속형 수치형 데이터 vs 범주형 데이터 → 포인트 이분 상관계수
_ ,p_value = pointbiserialr(train_df['FamilySize'], train_df['Survived'])
print(f'p-value : {p_value:.4f}')

p-value : 0.6199


- p-value가 0.05보다 작으므로 FamilySize와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각하지 못한다.
- FamilySize Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 없다.

### Pclass1

In [11]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱
crosstab = pd.crosstab(train_df['Pclass_1'],train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Pclass1과 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Pclass1과 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Pclass2

In [12]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['Pclass_2'],train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0069


- p-value가 0.05보다 작으므로 Pclass2와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Pclass2와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다..

### Pclass3

In [13]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['Pclass_3'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Pclass3와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Pclass3과 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Embarked_C

In [14]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱
crosstab = pd.crosstab(train_df['Embarked_C'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)
print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 EmbarkedC와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Embarked_C와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Embarked_Q

In [15]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱
crosstab = pd.crosstab(train_df['Embarked_Q'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 1.0000


- p-value가 0.05보다 작으므로 Embarked_Q와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각 할 수 없다.
- FamilySize Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 없다.

### Embarked_S

In [16]:
# 범주형 데이터 vs 범주형데이터 → 카이제곱
crosstab = pd.crosstab(train_df['Embarked_S'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Embarked_S와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- FamilySize와 Embarked_S는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Title_Master

In [17]:
# 범주형 데이터 vs 범주형 데이터 → 카이제곱
crosstab= pd.crosstab(train_df['Title_Master'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0174


- p-value가 0.05보다 작으므로 Title Master와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다..
- Title Master와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Title_Miss

In [18]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['Title_Miss'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Title_Miss와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- TitleMiss와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Title_Mr

In [19]:
# 범주형 데이터 vs 범주형 데이터 
crosstab = pd.crosstab(train_df['Title_Mr'],train_df['Survived'])
_, p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Title_Mr와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각 한다.
- Title_Mr Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Title_Mrs

In [20]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['Title_Mrs'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 Title_Mrs와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- Title_Mrs Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### Title_Rare

In [21]:
# 범주형데이터 vs 범주형데이터
crosstab = pd.crosstab(train_df['Title_Rare'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.8866


- p-value가 0.05보다 작으므로 Title_Rare와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각하지 못한다.
- Title_Rare와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 없다.

### FamilySizeGroup_0

In [22]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['FamilySizeGroup_0'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 보다 0.05보다 작으므로 FamilySizeGroup_0와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- FamilySizeGroup_0와 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### FamilySizeGroup_1

In [23]:
# 범주형 데이터 vs 범주형 데이터
crosstab = pd.crosstab(train_df['FamilySizeGroup_1'], train_df['Survived'])
_,p_value, _,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0000


- p-value가 0.05보다 작으므로 FamilySizeGroup_1와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- FamilySizeGroup_1 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### FamilySizeGroup_2

In [24]:
# 범주형 데이터 vs 범주형데이터
crosstab = pd.crosstab(train_df['FamilySizeGroup_2'], train_df['Survived'])
_,p_value,_,_ = chi2_contingency(crosstab)

print(f'p-value : {p_value:.4f}')

p-value : 0.0003


- p-value가 0.05보다 작으므로 FamilySizeGroup_2와 Survived는 서로 상관관계가 통계학적으로 무의미하다는 귀무가설을 기각한다.
- FamilySizeGroup_2 Survived는 서로 상관관계가 통계학적으로 유의미하다고 볼 수 있다.

### 상관관계가 낮은 컬럼들을 제거한다.

In [25]:
a1 = ['Age','SibSp', 'FamilySize', 'Embarked_Q', 'Title_Rare']
train_df.drop(a1, axis=1, inplace=True)
test_df.drop(a1, axis=1, inplace=True)

display(train_df)
display(test_df)

,Survived,Sex,Parch,Fare,isChild,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,FamilySizeGroup_0,FamilySizeGroup_1,FamilySizeGroup_2
0,0,1,0,2.110213,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1,0,0,4.280593,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,0,0,2.188856,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1,0,0,3.990834,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0,1,0,2.202765,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,1,0,2.639057,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
887,1,0,0,3.433987,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
888,0,0,2,3.196630,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
889,1,1,0,3.433987,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


,Sex,Parch,Fare,isChild,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,FamilySizeGroup_0,FamilySizeGroup_1,FamilySizeGroup_2
0,1,0,2.178064,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,0,0,2.079442,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,0,2.369075,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,1,0,2.268252,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0,1,2.586824,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1,0,2.202765,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
414,0,0,4.699571,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
415,1,0,2.110213,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
416,1,0,2.202765,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [26]:
train_df.to_csv('data/titanic_train5.csv')
test_df.to_csv('data/titanic_test5.csv')